Distortion Parameters:
 [[-3.30325200e-01  1.96909151e+00 -7.17910236e-03 -5.94455010e-04
  -4.54516181e+00]]

Camera Matrix:
 [[580.18421199   0.         306.32847071]
 [  0.         581.93758236 241.22427157]
 [  0.           0.           1.        ]]

NewCameraMatrix : [[470.08108521   0.         278.19215297]
 [  0.         520.09613037 233.85179557]
 [  0.           0.           1.        ]]

In [1]:
import numpy as np
import cv2
import sys
import time

In [2]:
def aruco_display(corners, ids, rejected, frame):
    image = frame.copy()
    ids = ids.flatten()
    
    for (markerCorner, markerID) in zip(corners, ids):
            
            corners = markerCorner.reshape((4, 2))
            (topLeft, topRight, bottomRight, bottomLeft) = corners
            
            topRight = (int(topRight[0]), int(topRight[1]))
            bottomRight = (int(bottomRight[0]), int(bottomRight[1]))
            bottomLeft = (int(bottomLeft[0]), int(bottomLeft[1]))
            topLeft = (int(topLeft[0]), int(topLeft[1]))

            cv2.line(image, topLeft, topRight, (0, 255, 0), 2)
            cv2.line(image, topRight, bottomRight, (0, 255, 0), 2)
            cv2.line(image, bottomRight, bottomLeft, (0, 255, 0), 2)
            cv2.line(image, bottomLeft, topLeft, (0, 255, 0), 2)
            
            cX = int((topLeft[0] + bottomRight[0]) / 2.0)
            cY = int((topLeft[1] + bottomRight[1]) / 2.0)
            cv2.circle(image, (cX, cY), 4, (0, 0, 255), -1)
            
            cv2.putText(image, str(markerID),(topLeft[0], topLeft[1] - 10), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (0, 255, 0), 2)
            # print("[Inference] ArUco marker ID: {}".format(markerID))
            
    return image

In [3]:
def pose_estimation(frame, aruco_dict_type, matrix_coefficients, distortion_coefficients):

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    cv2.aruco_dict = cv2.aruco.getPredefinedDictionary(aruco_dict_type)
    parameters = cv2.aruco.DetectorParameters()


    corners, ids, rejected = cv2.aruco.detectMarkers(gray, cv2.aruco_dict,parameters=parameters)

        
    if len(corners) > 0:
        for i in range(0, len(ids)):
           
            rvec, tvec, markerPoints = cv2.aruco.estimatePoseSingleMarkers(corners[i], 5, matrix_coefficients,
                                                                       distortion_coefficients)
            
#             detected_markers = aruco_display(corners, ids, rejected, frame)
#             cv2.imshow("Detected Markers", detected_markers) 


            cv2.putText(frame, f'X: {round(tvec[0][0][0],2)} Y: {round(tvec[0][0][1],2)} Z: {round(tvec[0][0][2],2)}',(50, 50), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (0, 255, 0), 2)

            cv2.drawFrameAxes(frame, matrix_coefficients, distortion_coefficients, rvec, tvec, 0.02)  

    return frame

In [4]:
# Parameters
arucoDict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
arucoParams = cv2.aruco.DetectorParameters()


intrinsic_camera = np.array(((580.18421199, 0., 306.32847071), (0., 581.93758236, 241.22427157), (0., 0., 1. )))
distortion = np.array((-3.30325200e-01, 1.96909151e+00, -7.17910236e-03, -5.94455010e-04, -4.54516181e+00))

In [5]:
intrinsic_camera

array([[580.18421199,   0.        , 306.32847071],
       [  0.        , 581.93758236, 241.22427157],
       [  0.        ,   0.        ,   1.        ]])

In [6]:
distortion

array([-3.30325200e-01,  1.96909151e+00, -7.17910236e-03, -5.94455010e-04,
       -4.54516181e+00])

In [7]:
intrinsic_camera.shape

(3, 3)

In [8]:
distortion.shape

(5,)

In [9]:
cap = cv2.VideoCapture(0)

while cap.isOpened():
    
    ret, img = cap.read()
    
    output = pose_estimation(img, cv2.aruco.DICT_4X4_50, intrinsic_camera, distortion)

    cv2.imshow('Estimated Pose', output)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [10]:
cap.release()
cv2.destroyAllWindows()